# Synthetic Data Auditor — full audit in Colab

Runs `sda_pipeline.py` for one dataset or the whole list in `datasets.json`:

1. clean → 80/20 members/holdout split (seed 42, stratified)
2. SDV GaussianCopula fitted on **members only**
3. fidelity (SDMetrics), utility (TRTR vs TSTR on the real holdout)
4. six privacy attacks: distance MIA, exact match, shadow-model MIA, singling-out, linkability, attribute inference (Anonymeter)

Each dataset produces one JSON file in `results/` in the same format the dashboard imports.


In [ ]:
!pip -q install sdv sdmetrics scikit-learn ucimlrepo
!pip -q install --no-deps anonymeter && pip -q install numba polars pyarrow joblib
!git clone -q https://github.com/vvrushtii/synthetic-data-auditor.git
%cd synthetic-data-auditor/pipeline

## Existing project datasets (cleaned CSVs from Drive)
Mount Drive and link the `original dataset` folder as `data/existing`. Skip this cell if you only run UCI datasets — those download automatically.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Change this path if the shared folder sits elsewhere in your Drive
SRC = '/content/drive/MyDrive/Synthetic_Data_Auditor/original dataset'
import os, shutil
os.makedirs('data/existing', exist_ok=True)
for src, dst in [('dataset1_adult','dataset1_adult.csv'), ('dataset2_wine_quality_clean.csv',None), ('dataset3_bank_marketing','dataset3_bank_marketing.csv'),
                 ('dataset5_breast cancer','dataset5_breast_cancer.csv'), ('dataset6_appliances_energy_clean.csv',None), ('dataset7_dry_bean_clean 3.csv','dataset7_dry_bean_clean_3.csv'),
                 ('dataset8_student_performance_math_clean.csv',None), ('dataset10_credit_card_default_clean.csv',None)]:
    p = os.path.join(SRC, src)
    if os.path.exists(p): shutil.copy(p, os.path.join('data/existing', dst or src)); print('copied', src)
    else: print('not found:', src)

## Run
`--name` runs one dataset; `--group new` runs the 30 new UCI datasets; no filter runs all 41. `--skip-done` resumes after a disconnect.

In [ ]:
!python sda_pipeline.py --config datasets.json --name "Heart Disease (Cleveland)" --out results

In [ ]:
# Everything (about 20-40 minutes on a standard Colab runtime)
# !python sda_pipeline.py --config datasets.json --out results --skip-done

In [ ]:
!python summarize.py results
import pandas as pd; pd.read_csv('results/summary.csv')

In [ ]:
!zip -qr results.zip results -x 'results/synthetic/*'
from google.colab import files; files.download('results.zip')